In [ ]:
from typing import Literal
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_core.prompts import PromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.runnables import RunnableBranch, RunnableLambda

load_dotenv() 

In [ ]:
class Feedback(BaseModel):
    sentiment: Literal['positive', 'negative'] = Field( description='Give the sentiment of feedback' )

pyd_output_parser = PydanticOutputParser(pydantic_object=Feedback)


model = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite"
)


prompt = PromptTemplate(
    template="""Extract the sentiment of the following feedback text into positive or negative \n{feedback} \n{formate_instructions}""",
    input_variables=['feedback'],
    partial_variables={'formate_instructions': pyd_output_parser.get_format_instructions()}
)

positive_feedback_prompt = PromptTemplate(
    template="""Write an appropriate feedback to following positive feedback. \n{feedback}""",
    input_variables=['feedback']
)

negative_feedback_prompt = PromptTemplate(
    template="""Write an appropriate feedback to following negative feedback. \n{feedback}""",
    input_variables=['feedback']
)


output_parser = StrOutputParser()

In [ ]:
classifier_chain = prompt | model | pyd_output_parser 

In [ ]:
# chain = RunnableBranch(
#     (condition1, chain),    # tuple
#     (condition2, chain),    # tuple
#     (default chain)         # tuple
# )


branch_chain = RunnableBranch(
    (lambda x: x.sentiment == 'positive', positive_feedback_prompt | model | output_parser),
    (lambda x: x.sentiment == 'negative', negative_feedback_prompt | model | output_parser),
    RunnableLambda(lambda x: "could not find sentiment")
)



final_chain = classifier_chain | branch_chain

final_chain.invoke({'feedback': 'The product is awsome.'})



In [ ]:
final_chain.get_graph().print_ascii()